In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("data\Placement_Data_Full_Class.csv")
df.columns

Index(['gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'hsc_s', 'degree_p',
       'degree_t', 'workex', 'etest_p', 'specialisation', 'mba_p', 'status',
       'salary'],
      dtype='object')

In [8]:
for col in df.columns:
    if df[col].dtype=='object':
        print(col,df[col].unique())

gender ['M' 'F']
ssc_b ['Others' 'Central']
hsc_b ['Others' 'Central']
hsc_s ['Commerce' 'Science' 'Arts']
degree_t ['Sci&Tech' 'Comm&Mgmt' 'Others']
workex ['No' 'Yes']
specialisation ['Mkt&HR' 'Mkt&Fin']
status ['Placed' 'Not Placed']


In [3]:
df.drop(['salary'],axis=1)
df.select_dtypes(include=['number']).columns

Index(['sl_no', 'ssc_p', 'hsc_p', 'degree_p', 'etest_p', 'mba_p', 'salary'], dtype='object')

In [15]:
df.drop(['salary'],axis=1)
df.select_dtypes(include=['object']).columns

Index(['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex',
       'specialisation', 'status'],
      dtype='object')

In [21]:
df["status"].unique()

array(['Placed', 'Not Placed'], dtype=object)

In [ ]:
df['status']=df["status"].str.replace({'Placed':1,'Not Placed':0})

0      False
1      False
2      False
3       True
4      False
       ...  
210    False
211    False
212    False
213    False
214     True
Name: status, Length: 215, dtype: bool

In [3]:
import os
os.getcwd()
os.chdir(r"c:\Users\Ayush Chauhan\Documents\udemy ML\ml-nlp\24-mlops\myMLOPS")

In [4]:
from src.components.data_transformation import DataIngestion,DataTransformation
obj =DataIngestion()
train_data_path,test_data_path,raw_data_path=obj.initiate_data_ingestion()
obj2=DataTransformation()
xtrain,xtest,_=obj2.initiate_data_transformation(train_path=train_data_path,test_path=test_data_path)

In [5]:
x_train=xtrain[:,:-1]
x_test=xtest[:,:-1]
y_train=xtrain[:,-1]
y_test=xtest[:,-1]

In [6]:
_,counts=np.unique(y_train,return_counts=True)
counts

array([ 50, 111], dtype=int64)

In [7]:
from sklearn.linear_model import LogisticRegression
lin_model=LogisticRegression(n_jobs=-1,class_weight='balanced',random_state=42)
lin_model.fit(x_train,y_train)
y_pred=lin_model.predict(x_test)


In [8]:
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

              precision    recall  f1-score   support

         0.0       0.67      0.82      0.74        17
         1.0       0.91      0.81      0.86        37

    accuracy                           0.81        54
   macro avg       0.79      0.82      0.80        54
weighted avg       0.83      0.81      0.82        54

[[14  3]
 [ 7 30]]


In [15]:
from sklearn.ensemble import RandomForestClassifier
RFC=RandomForestClassifier(n_jobs=-1)
RFC.fit(x_train,y_train)
y_pred=RFC.predict(x_test)


In [26]:
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_true=y_test,y_pred=y_pred,))

              precision    recall  f1-score   support

         0.0       1.00      0.65      0.79        17
         1.0       0.86      1.00      0.92        37

    accuracy                           0.89        54
   macro avg       0.93      0.82      0.86        54
weighted avg       0.90      0.89      0.88        54

[[11  6]
 [ 0 37]]


In [27]:
np.unique(y_train,return_counts=True)

(array([0., 1.]), array([ 50, 111], dtype=int64))

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
RFC=RandomForestClassifier()
params={ 'n_estimators':[100,200,500],
    'criterion':['gini', 'entropy', 'log_loss'],
    'max_depth':[3,4,5,6],
    "max_features":['sqrt', 'log2'],
    "class_weight":['balanced', 'balanced_subsample']  }

grid=GridSearchCV(estimator=RFC,param_grid=params,n_jobs=-1,verbose=2,scoring="roc_auc")
grid.fit(x_train,y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


GridSearchCV(estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'class_weight': ['balanced', 'balanced_subsample'],
                         'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [3, 4, 5, 6],
                         'max_features': ['sqrt', 'log2'],
                         'n_estimators': [100, 200, 500]},
             scoring='roc_auc', verbose=2)

In [ ]:
#this are for the best parameters for the model
grid.best_params_

{'class_weight': 'balanced_subsample',
 'criterion': 'entropy',
 'max_depth': 5,
 'max_features': 'log2',
 'n_estimators': 100}

In [14]:
grid.best_params_

{'class_weight': 'balanced',
 'criterion': 'entropy',
 'max_depth': 6,
 'max_features': 'sqrt',
 'n_estimators': 100}

In [11]:
y_pred=grid.predict(x_test)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
RFC=RandomForestClassifier(**{'class_weight': 'balanced_subsample',
 'criterion': 'log_loss',
 'max_depth': 10,
 'max_features': 'sqrt',
 'n_estimators': 100})
params={"class_weight":[{0:1.62 , 1: 0.725225},{0:1.65 , 1: 0.725225},{0:1.68 , 1: 0.725225},{0:1.68 , 1: 0.725225},{0:1.70 , 1: 0.725225}]}

grid=GridSearchCV(estimator=RFC,param_grid=params,n_jobs=-1,verbose=2,scoring="roc_auc")
grid.fit(x_train,y_train)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


GridSearchCV(estimator=RandomForestClassifier(class_weight='balanced_subsample',
                                              criterion='log_loss',
                                              max_depth=10),
             n_jobs=-1,
             param_grid={'class_weight': [{0: 1.62, 1: 0.725225},
                                          {0: 1.65, 1: 0.725225},
                                          {0: 1.68, 1: 0.725225},
                                          {0: 1.68, 1: 0.725225},
                                          {0: 1.7, 1: 0.725225}]},
             verbose=2)

In [29]:
grid.best_params_

{'class_weight': {0: 1.62, 1: 0.725225}}

In [15]:
y_pred=grid.predict(x_test)

In [12]:
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_true=y_test,y_pred=y_pred,))

              precision    recall  f1-score   support

         0.0       0.93      0.76      0.84        17
         1.0       0.90      0.97      0.94        37

    accuracy                           0.91        54
   macro avg       0.91      0.87      0.89        54
weighted avg       0.91      0.91      0.90        54

[[13  4]
 [ 1 36]]


In [ ]:
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_true=y_test,y_pred=y_pred,))


              precision    recall  f1-score   support

         0.0       0.87      0.76      0.81        17
         1.0       0.90      0.95      0.92        37

    accuracy                           0.89        54
   macro avg       0.88      0.86      0.87        54
weighted avg       0.89      0.89      0.89        54

[[13  4]
 [ 2 35]]


In [33]:
RFC=RandomForestClassifier(**{'class_weight': 'balanced_subsample',
 'criterion': 'log_loss',
 'max_depth': 10,
 'max_features': 'sqrt',
 'n_estimators': 100})
RFC.fit(x_train,y_train)
y_pred=RFC.predict(x_test)

In [34]:
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_true=y_test,y_pred=y_pred,))

              precision    recall  f1-score   support

         0.0       0.92      0.71      0.80        17
         1.0       0.88      0.97      0.92        37

    accuracy                           0.89        54
   macro avg       0.90      0.84      0.86        54
weighted avg       0.89      0.89      0.88        54

[[12  5]
 [ 1 36]]


In [36]:
import os 
os.getcwd()

'c:\\Users\\Ayush Chauhan\\Documents\\udemy ML\\ml-nlp\\24-mlops\\myMLOPS'

In [35]:
import pickle

with open("model.pkl", "wb") as file:
    pickle.dump(RFC, file)